# 03 Category Mapping

Apply the MVP food group set to cleaned Egyptian and USDA data.

Output: `data/processed/category_mapping_summary.parquet`

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'data' / 'processed'
EGYPTIAN_PATH = OUTPUT_DIR / 'egyptian_food_clean.parquet'
USDA_PATH = OUTPUT_DIR / 'usda_food_clean.parquet'
OUTPUT_PATH = OUTPUT_DIR / 'category_mapping_summary.parquet'

TARGET_GROUPS = [
    'Grains', 'Vegetables', 'Fruits', 'Protein', 'Dairy', 'Legumes',
    'Healthy Fats', 'Composite Dish', 'Beverages', 'Snacks'
]
TARGET_GROUP_SET = set(TARGET_GROUPS)

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}. Run earlier notebooks first.')

def normalize_group(value: object, name: str) -> str:
    if isinstance(value, str) and value in TARGET_GROUP_SET:
        return value
    text = name.lower()
    if any(k in text for k in ['rice', 'bread', 'wheat', 'corn', 'barley', 'pasta', 'macaroni', 'flour', 'grain']):
        return 'Grains'
    if any(k in text for k in ['tomato', 'potato', 'onion', 'carrot', 'spinach', 'vegetable']):
        return 'Vegetables'
    if any(k in text for k in ['apple', 'banana', 'orange', 'date', 'fruit', 'grape']):
        return 'Fruits'
    if any(k in text for k in ['meat', 'beef', 'chicken', 'fish', 'egg', 'lamb']):
        return 'Protein'
    if any(k in text for k in ['milk', 'cheese', 'yogurt', 'cream']):
        return 'Dairy'
    if any(k in text for k in ['bean', 'lentil', 'chickpea', 'pea']):
        return 'Legumes'
    if any(k in text for k in ['oil', 'ghee', 'fat', 'nut', 'seed']):
        return 'Healthy Fats'
    if any(k in text for k in ['juice', 'drink', 'tea', 'coffee']):
        return 'Beverages'
    if any(k in text for k in ['cake', 'sweet', 'biscuit', 'snack', 'chocolate']):
        return 'Snacks'
    return 'Composite Dish'

def load_clean(path: Path) -> pd.DataFrame:
    require_file(path)
    return pd.read_parquet(path, engine='pyarrow')

egyptian = load_clean(EGYPTIAN_PATH)
usda = load_clean(USDA_PATH)

for df in (egyptian, usda):
    df['food_group'] = [normalize_group(group, name) for group, name in zip(df['food_group'], df['food_name_en'])]

egyptian.to_parquet(EGYPTIAN_PATH, index=False, engine='pyarrow')
usda.to_parquet(USDA_PATH, index=False, engine='pyarrow')

combined = pd.concat([egyptian.assign(dataset='egyptian'), usda.assign(dataset='usda')], ignore_index=True)
summary = (
    combined.groupby(['dataset', 'food_group'], dropna=False)
    .size()
    .reset_index(name='row_count')
    .sort_values(['dataset', 'food_group'])
)
summary.to_parquet(OUTPUT_PATH, index=False, engine='pyarrow')

print(f'Egyptian rows updated: {len(egyptian)}')
print(f'USDA rows updated: {len(usda)}')
print(f'Output: {OUTPUT_PATH}')
print(summary)
